In [0]:
def write_post_attributes_table(spark, run_date, db, table_names):
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{post_attributes} PARTITION (partition_date = '{run_date}')
              with base_attributes as (
                select `id` as post_id,
                max(datediff(to_date('{run_date}'), to_date(from_unixtime(post_time / 1000)))) as days_since_publish,
                coalesce(last(post_type), 0) as post_type, --帖子类型：0-普通贴，1-活动贴，2-长文贴，3-转发贴
                max(if(is_essence >= 1, 1, 0)) as is_essence, --是否是精华帖
                coalesce(last(post_attribute)-1, 1) as is_ugc, --0:PGC 1:UGC
                coalesce(last(is_sink), 1) as is_sink --帖子是否下沉
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where is_hide = 0
                and audit_status = 1 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and publish_approval_status in (2,3) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
                and to_date(from_unixtime(post_time/1000)) between date_sub('{run_date}', 365) and '{run_date}' --only collect the latest one year posts
                group by 1
              ),
              media_attributes as (
                select post_id,
                max(if(`type` != 2, 1, 0)) as has_pic,
                max(if(`type` = 2, 1, 0)) as has_video 
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_media
                where delete_at is null
                group by 1
              ),
              post_views as (
                select post_id, count(*) as `views`
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_browse
                where to_date(from_unixtime(browse_time/1000)) = '{run_date}'
                and delete_at is null
                group by 1
              ),
              post_likes as (
                select post_id, count(*) as likes
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_like
                where to_date(from_unixtime(like_time/1000)) = '{run_date}'
                and delete_at is null
                group by 1
              ),
              post_comments as (
                select post_id, count(*) as comments
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_comment
                where to_date(from_unixtime(content_time/1000)) = '{run_date}'
                and is_hide = 0
                and audit_status = 1
                and delete_at is null
                group by 1
              ),
              post_shares as (
                select content_id as post_id,
                count(*) as `shares`
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_click_main_i_d
                where page_name in ('community_content_share', 'community_recommend_share','commnuity_moment_share', 'commnunity_group_moment_share')
                and element='share'
                and user_id is not null
                and dt = date_sub('{run_date}',1)
                group by 1
              ),
              post_dwell_time as (
                select content_id as post_id, sum(event_duration) as dwell_time
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_engagement_i_d
                where event_name = 'mystar_page_progress'
                and page_name = 'community_content'
                and content_id is not null
                and dt = date_sub('{run_date}',1) --only available after 2024-07-30
                group by 1
              )
              select ba.*, 
              coalesce(ma.has_pic, 0) as has_pic,
              coalesce(ma.has_video, 0) as has_video,
              coalesce(pv.views, 0) as `views`,
              coalesce(pl.likes, 0) as likes,
              coalesce(pc.comments, 0) as comments,
              coalesce(ps.shares, 0) as `shares`,
              coalesce(pdt.dwell_time, 0) as dwell_time
              from base_attributes ba
              left join media_attributes ma
              on ba.post_id = ma.post_id 
              left join post_views pv
              on ba.post_id = pv.post_id
              left join post_likes pl
              on ba.post_id = pl.post_id
              left join post_comments pc
              on ba.post_id = pc.post_id
              left join post_shares ps
              on ba.post_id = ps.post_id
              left join post_dwell_time pdt
              on ba.post_id = pdt.post_id
              """.format(db = db,
                         post_attributes = table_names["post_attributes"],
                         run_date = run_date))

In [0]:
def write_user_attributes_table(spark, run_date, db, table_names):
    spark.sql("""
              INSERT OVERWRITE TABLE {db}.{user_attributes} PARTITION (partition_date = '{run_date}')
                with base_attributes as (
                select customer_id as user_id, 
                max(months_between(to_date('{run_date}'), to_date(from_unixtime(if(register_time = 0, 1385913600000, register_time) / 1000)))) as months_since_registration, -- replace register_time = 0 by unix time of 2013-12-01 (the second earliest date)
                max(months_between(to_date('{run_date}'), to_date(from_unixtime(agree_community_time / 1000)))) as months_since_consent,
                last(`identity`) as `identity`, --用户身份： 0粉丝，1车主，2官方人员
                max(if(to_date(from_unixtime(koc_time / 1000)) = '{run_date}', 1, 0)) as is_koc,
                max(if(photo_url is not null, 1, 0)) as has_profile_photo,
                last(case when ip_address is null then 'unknown'
                    when ip_address RLIKE '[A-Za-z0-9]' then 'overseas'
                    else regexp_replace(ip_address, '[省市]', '')
                    end) as ip_province
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer
                where coalesce(to_date(from_unixtime(update_at / 1000)),to_date(from_unixtime(create_at / 1000))) <= '{run_date}'
                and delete_type = 0
                and `identity` != 2
                and agree_community_time is not null
                group by 1
                ),
                platform as (
                select user_id, 
                mode(coalesce(platform, 'unknown')) as platform
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_launch_terminate_i_d
                where user_id is not null
                and dt = date_sub('{run_date}', 1)
                group by 1
                ),
                user_vehicle as (
                select customer_id as user_id,
                last(dvm.series, true) as vehicle_series
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer_vehicle cv
                left join (
                    select distinct bm6, series 
                    from bz_cn_dl.dwh_dimensions.dim_vehicle_master 
                    where series is not null) dvm
                on substring(cv.baumuster, 1, 6) = dvm.bm6
                where to_date(from_unixtime(cv.delete_at/1000)) < '{run_date}'
                group by 1
                ),
                community_visits as (
                select user_id, 
                count(distinct session_id) as community_visits
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_show_i_d
                where page_name in ('community_recommend','community_moment','community_activity','community_group','community_information')
                and user_id is not null
                and dt = date_sub('{run_date}', 1)
                group by 1
                ),
                posts_published as (
                select customer_id as user_id, 
                count(distinct `id`) as posts_published
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post
                where to_date(from_unixtime(post_time/1000)) = '{run_date}' 
                and audit_status != 2 --审核状态：0-审核中，1-已通过，2-未过审，3-待人工
                and is_hide = 0  
                and publish_approval_status not in (0, 4) --发布批准状态：0-待提交，1-待审核，2-已通过，3-已发布，4-已驳回
                and delete_at is null
                group by 1
                ),
                posts_viewed as (
                select customer_id as user_id, 
                count(distinct post_id) as posts_viewed
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_browse
                where to_date(from_unixtime(browse_time/1000)) = '{run_date}'
                group by 1
                ),
                posts_liked as (
                select customer_id as user_id, 
                count(distinct post_id) as posts_liked
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_like
                where to_date(from_unixtime(like_time/1000)) = '{run_date}'
                and delete_at is null
                group by 1
                ),
                posts_commented as (
                select customer_id as user_id, 
                count(distinct post_id) as posts_commented
                from bz_cn_dl.datalake_community.dw_cust_community_tb_post_comment
                where to_date(from_unixtime(content_time/1000)) = '{run_date}'
                and delete_at is null
                group by 1
                ),
                posts_shared as (
                select user_id, 
                count(distinct content_id) as posts_shared
                from bz_cn_dl.datalake_mystar.dw_cust_dwd_mystar_app_page_click_main_i_d
                where page_name in ('community_content_share', 'community_recommend_share','commnuity_moment_share', 'commnunity_group_moment_share')
                and element = 'share'
                and user_id is not null
                and dt = date_sub('{run_date}', 1)
                group by 1
                ),
                users_followed as (
                select customer_id as user_id, 
                count(distinct passive_customer_id) as users_followed
                from bz_cn_dl.datalake_community.dw_cust_community_tb_customer_follows
                where to_date(from_unixtime(follow_time/1000)) = '{run_date}'
                and delete_at is null
                group by 1
                ),
                tribes_joined as (
                select customer_id as user_id,
                count(distinct tribe_id) as tribes_joined
                from bz_cn_dl.datalake_community.dw_cust_community_tb_tribe_customer
                where to_date(from_unixtime(join_tribe_time/1000)) = '{run_date}'
                and apply_state = 1
                and delete_at is null
                group by 1
                )
                select ba.*,
                coalesce(p.platform, 'unknown') as platform,
                coalesce(uv.vehicle_series, 'unknown') as vehicle_series,
                coalesce(cv.community_visits, 0) as community_visits,
                coalesce(ps.posts_published, 0) as posts_published,
                coalesce(pv.posts_viewed, 0) as posts_viewed,
                coalesce(pl.posts_liked, 0) as posts_liked,
                coalesce(pc.posts_commented, 0) as posts_commented,
                coalesce(psh.posts_shared, 0) as posts_shared,
                coalesce(uf.users_followed, 0) as users_followed,
                coalesce(tj.tribes_joined, 0) as tribes_joined
                from base_attributes ba
                left join platform p
                on ba.user_id = p.user_id
                left join user_vehicle uv
                on ba.user_id = uv.user_id
                left join community_visits cv
                on ba.user_id = cv.user_id
                left join posts_published ps
                on ba.user_id = ps.user_id
                left join posts_viewed pv
                on ba.user_id = pv.user_id
                left join posts_liked pl
                on ba.user_id = pl.user_id
                left join posts_commented pc
                on ba.user_id = pc.user_id
                left join posts_shared psh
                on ba.user_id = psh.user_id
                left join users_followed uf
                on ba.user_id = uf.user_id
                left join tribes_joined tj
                on ba.user_id = tj.user_id
              """.format(db = db,
                         user_attributes = table_names["user_attributes"],
                         run_date = run_date))

In [0]:
import json

# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
write_user_attributes_table(spark, run_date, db, table_names)

In [0]:
write_post_attributes_table(spark, run_date, db, table_names)